# 🏨 Hotel Reservation System 1: The Environment and the Coordinator

Notebooks 01–10 build one multi-agent system step by step: a hotel's customer-support desk where a
**conversation coordinator** talks to the guest and forwards work to specialist agents (reservations,
compliance, and later translation, sentiment and web search). This notebook sets up the world those agents live
in (a room database, the hotel's compliance rules and the agents' prompts), then builds the first agent, the
coordinator, and gives it a memory.

## Learning Objectives
1. **The room database** - A SQLite schema with rooms and reservations, seeded with dates relative to today
2. **Compliance rules as a vector store** - One document per rule, searchable through a `retrieve` tool
3. **The agents' prompts** - What `hotel_common.py` tells the coordinator, the reservation agent and the compliance checker
4. **A coordinator with memory** - A one-node graph that forgets the previous turn until a checkpointer and a
   `thread_id` give it one conversation per guest
5. **Forwarding by name** - How the coordinator will route by *naming* an agent, and why that is fragile

## Where this fits
Modules 05–09 covered multi-agent patterns in isolation. Module 10 applies them to complete systems, starting
with this ten-notebook build. Checkpointers and threads were taught in module 02's
[`01_memory_and_conversational_agent.ipynb`](../../m23_conversational_memory_rag/notebooks/06a_memory_and_conversational_agent.ipynb); here they give the coordinator its memory.

| Notebook | Adds |
|---|---|
| 02 | a reservation agent with SQL tools |
| 03 | a compliance checker over the rules: the core system the later notebooks extend |
| 04 | edge cases: injection, human review, past dates |
| 05 | testing the system, self-assessed confidence |
| 06 | parallel reservation agents over two hotels |
| 07 | merging agent outputs and summarizing long conversations |
| 08 | the guest's language and sentiment |
| 09 | system-wide error handling: retries and recovery |
| 10 | a web-search agent |

## Prerequisites
- Checkpointers and threads from module 02 (reading only; this notebook defines everything it uses)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the LangGraph building blocks and `hotel_common`, the shared
module next to this notebook.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import sqlite3
from datetime import date, timedelta

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

import hotel_common as hc  # shared hotel building blocks, in this folder
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the coordinator agent
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🗄️ Part 1: The Room Database

`init_hotel_db()` drops and recreates two tables every time it runs, so a re-run never duplicates rows:
- `rooms`: 8 rooms, from a 100-dollar Single to a 350-dollar Executive Suite
- `reservations`: 3 existing bookings, dated **relative to today** so the demos never drift into the past

Room 105 (the Family Suite) is booked from today+30 to today+33. Later notebooks use it to show a clash.

In [ ]:
# ============================================================================
# CREATE AND INSPECT THE DATABASE
# ============================================================================
db_path = hc.init_hotel_db("hotel.db")
print(f"✅ Database ready at {db_path}\n")

with sqlite3.connect("hotel.db") as conn:
    rooms = pd.read_sql("SELECT * FROM rooms", conn)
    reservations = pd.read_sql("SELECT * FROM reservations", conn)
print(rooms.to_string(index=False), "\n")
print(reservations.to_string(index=False))

print("\nSchema given to the SQL agent:\n" + hc.SQL_SCHEMA)

---
## 📜 Part 2: The Compliance Rules

The compliance checker of `03_hotel_compliance_checker.ipynb` must decide whether a request breaks a hotel
rule. Each rule is indexed as its **own** document in an in-memory Chroma collection (OpenAI `text-embedding-3-small` embeddings), so a search
returns the specific rule that applies rather than one blob of text.

In [ ]:
# ============================================================================
# INDEX THE RULES AND TEST THE RETRIEVER
# ============================================================================
for i, rule in enumerate(hc.COMPLIANCE_RULES, start=1):
    print(f"Rule {i}: {rule}")

retrieve = hc.build_compliance_retriever()
print("\n🔎 retrieve('Which dates is Bob Johnson staying?')\n")
print(retrieve.invoke({"query": "Which dates is Bob Johnson staying?"}))

**Reading the output:** the top hit should be the rule about other guests' information. That is the rule the
compliance checker will need to cite when a guest asks about someone else's stay.

---
## 🗒️ Part 3: The Agents' Prompts

Every agent in this series takes its instructions from `hotel_common.py`, so the later notebooks import them
instead of re-typing them. They are printed here once, so you know exactly what each agent is told:
- **`hc.conversation_prompt(agents, extra_guidelines)`**: the coordinator. It lists the agents it may forward to
  and forwards by writing an agent's exact name (Part 6). `hc.COMPLIANCE_FIRST` is the guideline
  `03_hotel_compliance_checker.ipynb` adds: every new request goes to the compliance checker first.
- **`hc.SQL_PROMPT`**: the reservation agent of `02_hotel_reservation_agent.ipynb`. It reads with `run_query`,
  books with `book_room`, and is told that guest names are private. (`hc.SQL_READ_PROMPT` is the same without
  booking, for the parallel agents of `06_hotel_parallel_reservation_agents.ipynb`.)
- **`hc.COMPLIANCE_PROMPT`**: the compliance checker. It must call `retrieve` and answer `COMPLIANT` or
  `VIOLATION`, citing only the rules it retrieved.

The checks at the end confirm in code that each prompt contains the part the later notebooks rely on.

In [ ]:
# ============================================================================
# THE THREE SYSTEM PROMPTS, AS THE MODELS SEE THEM
# ============================================================================
AGENTS = ["reservation_assistant", "compliance_checker"]
prompts = {
    "Coordinator: hc.conversation_prompt(AGENTS, extra_guidelines=hc.COMPLIANCE_FIRST)":
        hc.conversation_prompt(AGENTS, extra_guidelines=hc.COMPLIANCE_FIRST),
    "Reservation agent: hc.SQL_PROMPT": hc.SQL_PROMPT,
    "Compliance checker: hc.COMPLIANCE_PROMPT": hc.COMPLIANCE_PROMPT,
}
for label, prompt in prompts.items():
    print(f"━━━ {label} ━━━\n{prompt.content}\n")

coordinator, sql, compliance = (prompt.content for prompt in prompts.values())
checks = [
    ("the coordinator lists both agents by name", all(f"- {agent}" in coordinator for agent in AGENTS)),
    ("the coordinator sends every new request to compliance_checker first", hc.COMPLIANCE_FIRST in coordinator),
    ("the reservation agent books only through book_room", "book_room(" in sql),
    ("the reservation agent is told that guest names are private", "Guest names are private" in sql),
    ("the compliance checker must call retrieve and answer COMPLIANT or VIOLATION",
     "`retrieve`" in compliance and "COMPLIANT" in compliance and "VIOLATION" in compliance),
]
for label, ok in checks:
    print(f"{'✅' if ok else '❌'} {label}")

---
## 🤵 Part 4: The Coordinator Graph

The first agent is the **conversation coordinator**: the one that greets the guest and, from the next notebook
on, forwards work to specialists with the prompt printed above. There are no specialists yet, so here it gets a
shorter prompt that only asks it to engage the guest and gather what a booking needs. On its own it is a
one-node graph over `MessagesState`: `START → conv_assistant → END`.

In [ ]:
# ============================================================================
# COORDINATOR PROMPT, NODE AND GRAPH
# ============================================================================
coordinator_prompt = SystemMessage(content=f"""Role: You are the Conversation Coordinator for a hotel chain's customer support system.
Engage the guest politely, find out what they need, and keep a friendly, empathetic tone.
To book a room you need the room type, the dates and the guest's name: ask for whatever is missing.
The hotel is at {hc.HOTEL_ADDRESS}.""")


def conv_assistant(state: MessagesState) -> dict:
    return {"messages": [llm.invoke([coordinator_prompt] + state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("conv_assistant", conv_assistant)
builder.add_edge(START, "conv_assistant")
builder.add_edge("conv_assistant", END)

graph = builder.compile()
print("✅ Graph compiled. Edges:")
hc.print_edges(graph)

---
## 🧠 Part 5: Memory Without and With a Checkpointer

Module 02's [`01_memory_and_conversational_agent.ipynb`](../../m23_conversational_memory_rag/notebooks/06a_memory_and_conversational_agent.ipynb) teaches checkpointers in depth; this is the
short version, on the coordinator. The guest sends two messages: a booking request, then the dates.
- **Without a checkpointer** every `invoke` starts from an empty state, so the second message reaches the model
  alone.
- **With `InMemorySaver()`** the graph stores its state after every step under the `thread_id` in the config,
  and the next call on that thread starts from it. `hc.run_turn()` sends one message on a thread and prints what
  the agent added.

The dates are computed from today, so they are always in the future.

In [ ]:
# ============================================================================
# TWO TURNS, NO CHECKPOINTER
# ============================================================================
check_in = date.today() + timedelta(days=60)
check_out = check_in + timedelta(days=4)
turn_1 = "Hello, I want to book a room in your hotel!"
turn_2 = f"From {check_in:%d %B} till {check_out:%d %B}."

for text in (turn_1, turn_2):
    result = graph.invoke({"messages": [HumanMessage(content=text)]})
    seen = len(result["messages"]) - 1  # everything except the reply: what the model was sent
    print(f"👤 {text}\n🤵 {result['messages'][-1].content}\n   (messages the model saw: {seen})\n")

print(f"{'✅' if seen == 1 else '❌'} without a checkpointer the second turn reached the model as {seen} message: "
      "the dates, alone")

In [ ]:
# ============================================================================
# TWO TURNS ON ONE THREAD, THEN A SECOND GUEST
# ============================================================================
graph_with_memory = builder.compile(checkpointer=InMemorySaver())

hc.run_turn(graph_with_memory, turn_1, thread_id="guest-1")
state = hc.run_turn(graph_with_memory, turn_2, thread_id="guest-1")
other = hc.run_turn(graph_with_memory, "Which dates did I just give you?", thread_id="guest-2")

stored_1, stored_2 = len(state["messages"]), len(other["messages"])
print(f"\n{'✅' if stored_1 == 4 else '❌'} thread guest-1 stores {stored_1} messages: both turns, so the model "
      "saw the dates together with the booking request")
print(f"{'✅' if stored_2 == 2 else '❌'} thread guest-2 stores {stored_2} messages: only its own turn, so it "
      "cannot know guest-1's dates")

**Reading the output:** without a checkpointer the second turn reached the model as **1** message, the dates
with no booking around them. With `InMemorySaver`, thread `guest-1` holds 4 messages and the reply to the dates
treats them as part of the booking. Thread `guest-2` holds only its own 2 messages, so that guest's question
cannot be answered: the threads are isolated.

To chat with the agent yourself, run `hc.chat_with_agent(graph_with_memory, "my-thread")` in a new cell
(type `quit` to stop).

---
## 🔀 Part 6: How the Coordinator Will Forward

From the next notebook on, the coordinator forwards the conversation by **naming** an agent in its reply, and a
small function checks the reply for those names. It is easy to read and easy to break: any reply that merely
*mentions* an agent routes there too.

The series keeps this string routing on purpose, so the weak spot stays visible:
`03_hotel_compliance_checker.ipynb` adds a loop guard to it (`hc.route_by_mention()`), and
`05_hotel_testing_the_system.ipynb` pins the false positive below down as a known issue in its router tests. The
robust fix is a **typed router**: the model returns its choice as a structured-output field, so there is no
reply text to search. `13_multi_agent_research_summarization.ipynb` routes that way, as does Part 1 of module
08's `05_meta_controller.ipynb`.

In [ ]:
# ============================================================================
# STRING ROUTING, INCLUDING ITS WEAK SPOT
# ============================================================================
agents = ["reservation_assistant", "compliance_checker"]
examples = [
    "Let me forward you to the reservation_assistant.",
    "I'll pass this to our compliance checker first.",
    "Our breakfast is served from 7 to 10.",
    "I can't involve the reservation assistant until you give me dates.",  # mentions it, but should NOT route
]
for reply in examples:
    print(f"{str(hc.mentioned_agent(reply, agents)):<22} <- {reply}")

false_positive = hc.mentioned_agent(examples[-1], agents)
print(f"\n{'⚠️' if false_positive else '✅'} the last reply routes to {false_positive}, although it says it "
      "will NOT forward")

**Reading the output:** the last reply routes to `reservation_assistant` although it says the opposite (the ⚠️
line). Keep that weak spot in mind: the coordinator's prompt tells it to mention an agent only to forward, but
nothing *forces* it.

---
## 📝 Summary

### 1. The world the agents live in
- `hc.init_hotel_db()` rebuilds the rooms/reservations database with bookings relative to today;
  `hc.build_compliance_retriever()` indexes one document per rule and returns a `retrieve` tool.

### 2. The agents' prompts
- `hc.conversation_prompt()`, `hc.SQL_PROMPT` and `hc.COMPLIANCE_PROMPT` are printed above; every later notebook
  imports them from `hotel_common.py`.

### 3. A coordinator with memory
- A one-node graph over `MessagesState` forgets the previous turn. `InMemorySaver` + `thread_id` keeps one
  conversation per guest, and separate threads stay isolated.

### 4. Forwarding by name
- `hc.mentioned_agent()` routes on the agent a reply names: simple, and fooled by a mere mention.
- `hc.run_turn()` sends one scripted guest message and prints what the agents did; `hc.chat_with_agent()` is the
  interactive version for your own experiments.

### Next Steps
- `02_hotel_reservation_agent.ipynb`: the first specialist, a reservation agent that queries and books rooms with
  SQL tools